In [ ]:
import torch

In [ ]:
x = torch.tensor(3.0, requires_grad = True)

In [ ]:
y = x**2

In [ ]:
print(x)
print(y)

tensor(3., requires_grad=True)
tensor(9., grad_fn=<PowBackward0>)


In [ ]:
y.backward() # ------> calculated derivative

In [ ]:
x.grad # --------> 2x == 2*3 ==> 6 will be the output

tensor(6.)

### let'stake a bigger example with chain rule

In [ ]:
x = torch.tensor(3.0, requires_grad = True)

In [ ]:
y = x ** 2

In [ ]:
z = torch.sin(y)

In [ ]:
print(x)
print(y)
print(z)

tensor(3., requires_grad=True)
tensor(9., grad_fn=<PowBackward0>)
tensor(0.4121, grad_fn=<SinBackward0>)


In [ ]:
z.backward()

In [ ]:
x.grad

tensor(-5.4668)

In [ ]:
y.grad

/tmp/ipykernel_1783/486760323.py:1: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:494.)
  y.grad


### Let's do it on simple Neural **network**

In [ ]:
import torch

x = torch.tensor(6.7)
y = torch.tensor(0.0)

w = torch.tensor(1.0)
b = torch.tensor(0.0)


In [ ]:
# Binary cross entropy loss for scalar
def binary_cross_entropy_loss(prediction, target):
  eps = 1e-8
  prediction = torch.clamp(prediction, eps, 1 - eps)
  return -(target * torch.log(prediction) + (1 - target) * torch.log(1 - prediction))

In [ ]:
# forward pass
z = w * x + b
y_pred = torch.sigmoid(z)

# compute binary cross entropy loss
loss = binary_cross_entropy_loss(y_pred, y)

In [ ]:
loss

tensor(6.7012)

In [ ]:
# derivatives

# 1. dl / d y_pred
dl_dy_pred = (y_pred - y) / (y_pred * (1 - y_pred))

# 2. d y_pred/ dz
dy_pred_dz = y_pred * (1 - y_pred)

dz_dw = x
dz_db = 1

dL_dw= dl_dy_pred * dy_pred_dz * dz_dw

dL_dw = dl_dy_pred * dy_pred_dz * dz_dw
dL_db = dl_dy_pred * dy_pred_dz * dz_db

In [ ]:
print(f"manual gradient of loss w.r.t weight (dw): ", {dL_dw})
print(f"manual gradient of loss w.r.t bias (db): ", {dL_db})

manual gradient of loss w.r.t weight (dw):  {tensor(6.6918)}
manual gradient of loss w.r.t bias (db):  {tensor(0.9988)}


### Do it by using tensors

In [ ]:
x = torch.tensor(6.7)
y = torch.tensor(0.0)

w = torch.tensor(1.0, requires_grad = True)
b = torch.tensor(0.0, requires_grad = True)

In [ ]:
print(w)
print(b)

tensor(1., requires_grad=True)
tensor(0., requires_grad=True)


In [ ]:
z = w * x + b

In [ ]:
y_pred = torch.sigmoid(z)

In [ ]:
y_pred

tensor(0.9988, grad_fn=<SigmoidBackward0>)

In [ ]:
loss = binary_cross_entropy_loss(y_pred, y)

In [ ]:
loss

tensor(6.7012, grad_fn=<NegBackward0>)

In [ ]:
loss.backward()

In [ ]:
print(w.grad)
print(b.grad)

tensor(6.6918)
tensor(0.9988)


### calculating grads for multiple values simulteneously

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad = True)

In [ ]:
y = (x ** 2).mean()
y

tensor(4.6667, grad_fn=<MeanBackward0>)

In [ ]:
y.backward()

In [ ]:
x.grad

tensor([0.6667, 1.3333, 2.0000])

### Clearing Gradients

In [ ]:
x = torch.tensor(2.0, requires_grad= True)

In [ ]:
x

tensor(2., requires_grad=True)

In [ ]:
y = x ** 2
y

tensor(4., grad_fn=<PowBackward0>)

In [ ]:
y.backward()

In [ ]:
x.grad # If you run same code again then instead or overriding value it will add it
        # if x.grad == 4 then next time it will be 8 and 12, 16, 20 and so on

tensor(8.)

In [ ]:
# so clearing this values is so important

x.grad.zero_()

tensor(0.)

### **How to desable Gradient**

In [ ]:
x.requires_grad_(False)

tensor(2.)

In [ ]:
x   # requires_grad will not be visible as it is desable now

tensor(2.)

In [ ]:
y = x ** 2
y

tensor(4.)

In [ ]:
y.backward() # will not gonna work as x has now grad enabled

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [ ]:
# 2nd option isdetach()

x = torch.tensor(2.0, requires_grad = True)
x

tensor(2., requires_grad=True)

In [ ]:
z = x.detach()  # so z is clone but does not hav grad

In [ ]:
z

tensor(2.)

In [ ]:
y = x**2  # this will have backward and grad enabled
y

tensor(4., grad_fn=<PowBackward0>)

In [ ]:
y1 = z**2 # this will not have backward and grad
y1

tensor(4.)

In [ ]:
y.backward()

In [ ]:
y1.backward()

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

### torch.no_grad()

In [ ]:
# most convinent way of this is this :)
x = torch.tensor(2.0, requires_grad = True)
x

tensor(2., requires_grad=True)

In [ ]:
with torch.no_grad():
  y = x**2

In [ ]:
y # disabled

tensor(4.)